### **Dataframe Chunking 1**

See python script.

### **Dataframe Chunking 2**

We read this line `/usr/bin/time -l python dataframe_chunking_1.py data/2023_01.csv 1000`, which yields:
```bash
12548.630000000054
       12.33 real        10.18 user         2.03 sys
           125616128  maximum resident set size
                   0  average shared memory size
                   0  average unshared data size
                   0  average unshared stack size
              629061  page reclaims
                 148  page faults
                   0  swaps
                   0  block input operations
                   0  block output operations
                   0  messages sent
                   0  messages received
                   0  signals received
                  14  voluntary context switches
                5558  involuntary context switches
        110211977194  instructions retired
         39161580610  cycles elapsed
            87033408  peak memory footprint
```

We see 125616128/1024 = 122880 KB memory and run time is 12.33 seconds.

Nect, for 10000 we have:
```bash
12548.629999999994
        7.06 real         6.43 user         0.50 sys
           136445952  maximum resident set size
                   0  average shared memory size
                   0  average unshared data size
                   0  average unshared stack size
               98095  page reclaims
                  21  page faults
                   0  swaps
                   0  block input operations
                   0  block output operations
                   0  messages sent
                   0  messages received
                   0  signals received
                 470  voluntary context switches
                3351  involuntary context switches
         75111763368  instructions retired
         22168072124  cycles elapsed
            98190976  peak memory footprint
```
So we have 133248 KB and run time is 7.06 seconds.

Next for 100000 we have:
```bash
12548.629999999997
        6.73 real         5.88 user         0.56 sys
           247447552  maximum resident set size
                   0  average shared memory size
                   0  average unshared data size
                   0  average unshared stack size
              149811  page reclaims
                2447  page faults
                   0  swaps
                   0  block input operations
                   0  block output operations
                   0  messages sent
                   0  messages received
                   0  signals received
                 470  voluntary context switches
                4740  involuntary context switches
         71779155410  instructions retired
         20612069202  cycles elapsed
           201148352  peak memory footprint
```
So we have 241648 KB and run time is 6.73 seconds.

Finally, for 1000000 we have:
```bash
12548.630000000001
        6.70 real         6.00 user         0.46 sys
           644235264  maximum resident set size
                   0  average shared memory size
                   0  average unshared data size
                   0  average unshared stack size
              106920  page reclaims
                  89  page faults
                   0  swaps
                   0  block input operations
                   0  block output operations
                   0  messages sent
                   0  messages received
                   0  signals received
                 470  voluntary context switches
                8716  involuntary context switches
         70791306068  instructions retired
         20503343432  cycles elapsed
           605391552  peak memory footprint
```
So we have 629136 KB and run time is 6.70 seconds.

### **Dataframe Chunking 3**

In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

input_file = "data/2023_01.csv"
output_file = "data/2023_01.parquet"

chunk_size = 100_000

writer = None

for chunk in pd.read_csv(input_file, chunksize=chunk_size):
    table = pa.Table.from_pandas(chunk)

    if writer is None:
        writer = pq.ParquetWriter(output_file, table.schema)

    writer.write_table(table)

if writer:
    writer.close()

### **Dataframe Chunking 4**

We see that already for 1000 it is much faster:
```bash
        0.12 real         0.08 user         0.02 sys
            51265536  maximum resident set size
                   0  average shared memory size
                   0  average unshared data size
                   0  average unshared stack size
                3748  page reclaims
                  31  page faults
                   0  swaps
                   0  block input operations
                   0  block output operations
                   0  messages sent
                   0  messages received
                   0  signals received
                 166  voluntary context switches
                  99  involuntary context switches
           793532243  instructions retired
           311554509  cycles elapsed
            25658880  peak memory footprint
```
For instance, we now have 0.12 seconds as run time. Parquet improves performance due to reduced parsing and better data layout.

### **Dataframe Chunking 5**

The script should be modified to:
```python
import sys
import pyarrow.parquet as pq

def compute_precipitation(filename, chunk_size):
    parquet_file = pq.ParquetFile(filename)

    total_precip = 0

    for batch in parquet_file.iter_batches(batch_size=chunk_size, columns=['parameterId', 'value']):
        data = batch.to_pydict()

        param = data['parameterId']
        values = data['value']

        total_precip += sum(v for p, v in zip(param, values) if p == 'precip_past10min')

    return total_precip


filename = sys.argv[1]
chunk_size = int(sys.argv[2])

result = compute_precipitation(filename, chunk_size)
print(result)
```

### **Mandelbrot memmap 1**

See python script.

### **Mandelbrot memmap 2**

### **Mandelbrot memmap 3**

### **Mandelbrot memmap 4**

### **Mandelbrot Zarr 1**

Removed.

### **Mandelbrot Zarr 2**

Removed.

### **Mandelbrot Zarr 3**

Removed.

### **Mandelbrot Zarr 4**

Removed.